In [ ]:
from project_config import CATALOG_ROOT, RADAR_DOC_ROOT, NWP_DOC_ROOT, OUTPUT_ROOT, CITY_BOUNDARY_PATH, CASE_SPLIT, CASE_ID, CASE_INDEX, CASE_TITLE, WEIGHTED_HAILRU_CHECKPOINT, DIFF_T_CHECKPOINT, UNET_CHECKPOINT, load_pickle
from pathlib import Path
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
import torch
import torch.nn as nn
import torch.nn.functional as F
import pytorch_lightning as pl
import numpy as np
import os
import datetime
import matplotlib.pyplot as plt
import pandas as pd


In [ ]:
sta_dic = {}
data_root = RADAR_DOC_ROOT
file_list = os.listdir(data_root)
for i in range(len(file_list)):
    textid = file_list[i].split('.')[0].split('_')[2]
    sta_file = os.path.join(data_root, file_list[i])
    labelfmt = pd.read_csv(sta_file)
    lonss = labelfmt.iloc[0, 0].split('=')
    lons = round(float(lonss[1]), 4)
    lonee = labelfmt.iloc[2, 0].split('=')
    lone = round(float(lonee[1]), 4)
    latss = labelfmt.iloc[1, 0].split('=')
    lats = round(float(latss[1]), 4)
    latee = labelfmt.iloc[3, 0].split('=')
    late = round(float(latee[1]), 4)
    sta_dic[textid] = [lons, lone, lats, late]
sta_dic_nwp = {}
data_root = NWP_DOC_ROOT
file_list = os.listdir(data_root)
for i in range(len(file_list)):
    textid = file_list[i].split('.')[0].split('_')[2]
    sta_file = os.path.join(data_root, file_list[i])
    labelfmt = pd.read_csv(sta_file)
    lonss = labelfmt.iloc[0, 0].split('=')
    lons = round(float(lonss[1]), 4)
    lonee = labelfmt.iloc[2, 0].split('=')
    lone = round(float(lonee[1]), 4)
    latss = labelfmt.iloc[1, 0].split('=')
    lats = round(float(latss[1]), 4)
    latee = labelfmt.iloc[3, 0].split('=')
    late = round(float(latee[1]), 4)
    sta_dic_nwp[textid] = [lons, lone, lats, late]

def get_coord(stanum, coord, sta_dic, test=0):
    resol = 0.01
    [lons, lone, lats, late] = sta_dic[stanum]
    pix_lon = (coord[0] - lons) // resol
    pix_lat = 400 - (coord[1] - late) // resol
    pix_c = [int(pix_lon), int(pix_lat)]
    if test == 1:
        print('Grid coordinates', pix_c)
        print('Extent', lons, lone, lats, late)
        x1 = np.linspace(lons, lone, 401)
        y1 = np.linspace(late, lats, 401)[::-1]
        x1 = np.round(x1, 4)
        y1 = np.round(y1, 4)
        print('Input coordinates', coord)
        print('Matched coordinates', x1[pix_c[0]], y1[pix_c[1]])
        return (pix_c, [lons, lone, lats, late])
    elif 0 <= pix_c[0] < 401 and 0 <= pix_c[1] < 401:
        return pix_c
    else:
        return


In [ ]:
setname = CASE_SPLIT
data_dic = load_pickle(CATALOG_ROOT / setname / 'data_dic_nwp.pkl')
mask_dic = load_pickle(CATALOG_ROOT / setname / 'mask_dic_all.pkl')
train_dic_matrix = load_pickle(CATALOG_ROOT / setname / 'train_dic_matrix.pkl')


In [ ]:
class SpatialAttention(nn.Module):

    def __init__(self):
        super(SpatialAttention, self).__init__()
        self.conv = nn.Conv2d(2, 1, 7, 1, 3)
        self.act = nn.Sigmoid()

    def forward(self, x):
        maxpool = torch.max(x, dim=1, keepdim=True)[0]
        avgpool = torch.mean(x, dim=1, keepdim=True)
        SA = self.act(self.conv(torch.cat((maxpool, avgpool), dim=1)))
        return SA * x

class ChannelAttention(nn.Module):

    def __init__(self, dim):
        super(ChannelAttention, self).__init__()
        self.avgpool = nn.AdaptiveAvgPool2d(1)
        self.maxpool = nn.AdaptiveMaxPool2d(1)
        self.conv_shared = nn.Sequential(nn.Conv2d(dim, dim // 16, 1, 1), nn.LeakyReLU(), nn.Conv2d(dim // 16, dim, 1, 1))
        self.act = nn.Sigmoid()

    def forward(self, x):
        maxpool = self.conv_shared(self.maxpool(x))
        avgpool = self.conv_shared(self.avgpool(x))
        CA = self.act(maxpool + avgpool)
        return CA * x

class CBAM(nn.Module):

    def __init__(self, dim):
        super(CBAM, self).__init__()
        self.CA = ChannelAttention(dim)
        self.SA = SpatialAttention()

    def forward(self, x):
        x = self.CA(x)
        x = self.SA(x)
        return x

class IRCBAM(nn.Module):

    def __init__(self, inchannels):
        super(IRCBAM, self).__init__()
        self.c = inchannels
        self.act = nn.LeakyReLU()
        self.branch1 = nn.Sequential(nn.Conv2d(self.c, self.c // 4, 1, 1, 'same'), nn.BatchNorm2d(self.c // 4), self.act)
        self.branch2 = nn.Sequential(nn.Conv2d(self.c, self.c // 4, 1, 1, 'same'), nn.BatchNorm2d(self.c // 4), self.act, nn.Conv2d(self.c // 4, self.c // 4, 3, 1, 'same'), nn.BatchNorm2d(self.c // 4), self.act)
        self.branch3 = nn.Sequential(nn.Conv2d(self.c, self.c // 4, 1, 1, 'same'), nn.BatchNorm2d(self.c // 4), self.act, nn.Conv2d(self.c // 4, self.c // 4, 3, 1, 'same'), nn.BatchNorm2d(self.c // 4), self.act, nn.Conv2d(self.c // 4, self.c // 4, 3, 1, 'same'), nn.BatchNorm2d(self.c // 4), self.act)
        self.branch4 = nn.Sequential(nn.AvgPool2d(3, 1, 1), nn.Conv2d(self.c, self.c // 4, 1, 1, 'same'), nn.BatchNorm2d(self.c // 4), self.act)
        self.CBAM = CBAM(inchannels)

    def forward(self, x):
        identity = x
        b1 = self.branch1(x)
        b2 = self.branch2(x)
        b3 = self.branch3(x)
        b4 = self.branch4(x)
        cat = torch.cat([b1, b2, b3, b4], dim=1)
        cat = self.CBAM(cat)
        output = 0.3 * cat + identity
        return output

class HRUnet(nn.Module):

    def __init__(self, in_channels, out_channels):
        super(HRUnet, self).__init__()
        chans = [64, 256, 1024]
        ks = 3
        self.act = nn.LeakyReLU(inplace=True)
        self.act_output = nn.Sigmoid()
        self.channel_weights = nn.Parameter(torch.ones(1, in_channels, 1, 1))
        self.start_conv = nn.Sequential(nn.Conv2d(in_channels, chans[0], 2, 1, 0), nn.BatchNorm2d(chans[0]), self.act)
        self.ResBlock_11 = IRCBAM(chans[0])
        self.ResBlock_12 = IRCBAM(chans[0])
        self.DownConv_1 = nn.PixelUnshuffle(2)
        self.ResBlock_21 = IRCBAM(chans[1])
        self.ResBlock_22 = IRCBAM(chans[1])
        self.DownConv_2 = nn.PixelUnshuffle(2)
        self.ResBlock_31 = IRCBAM(chans[2])
        self.ResBlock_32 = IRCBAM(chans[2])
        self.ResBlock_33 = IRCBAM(chans[2])
        self.ResBlock_34 = IRCBAM(chans[2])
        self.UpConv_1 = nn.PixelShuffle(2)
        self.UpConv_next_1 = nn.Sequential(nn.Conv2d(chans[1] * 2, chans[1], ks, 1, 1), nn.BatchNorm2d(chans[1]), self.act)
        self.ResBlock_23 = IRCBAM(chans[1])
        self.ResBlock_24 = IRCBAM(chans[1])
        self.UpConv_2 = nn.PixelShuffle(2)
        self.UpConv_next_2 = nn.Sequential(nn.Conv2d(chans[0] * 2, chans[0], ks, 1, 1), nn.BatchNorm2d(chans[0]), self.act)
        self.ResBlock_13 = IRCBAM(chans[0])
        self.ResBlock_14 = IRCBAM(chans[0])
        self.final_conv = nn.Sequential(nn.Conv2d(chans[0], out_channels, 2, 1, 1), self.act_output)

    def forward(self, x, mask):
        x = x * self.channel_weights
        x = self.start_conv(x)
        x = self.ResBlock_11(x)
        x = self.ResBlock_12(x)
        x_pass_1 = x
        x = self.DownConv_1(x)
        x = self.ResBlock_21(x)
        x = self.ResBlock_22(x)
        x_pass_2 = x
        x = self.DownConv_2(x)
        x = self.ResBlock_31(x)
        x = self.ResBlock_32(x)
        x = self.ResBlock_33(x)
        x = self.ResBlock_34(x)
        x = self.UpConv_1(x)
        x = torch.cat((x, x_pass_2), dim=1)
        x = self.UpConv_next_1(x)
        x = self.ResBlock_23(x)
        x = self.ResBlock_24(x)
        x = self.UpConv_2(x)
        x = torch.cat((x, x_pass_1), dim=1)
        x = self.UpConv_next_2(x)
        x = self.ResBlock_13(x)
        x = self.ResBlock_14(x)
        x = self.final_conv(x)
        x = x * mask
        return x

class Loss(nn.Module):

    def __init__(self, alpha, beta):
        super(Loss, self).__init__()
        self.alpha = alpha
        self.beta = beta

    def forward(self, y_pred, y_true):
        base_temp = torch.square(y_pred - y_true) * ((y_true + self.alpha) / (1 + self.alpha))
        if torch.sum(y_true) > 0:
            temp = base_temp
        else:
            temp = self.beta * base_temp
        divisor = temp.shape[1] * temp.shape[2] * temp.shape[3]
        return torch.sum(temp) / divisor

class LightningModel(pl.LightningModule):

    def __init__(self, alpha, beta):
        super().__init__()
        self.model = HRUnet(102, 20)
        self.criterion = Loss(alpha, beta)

    def forward(self, x, mask):
        return self.model(x, mask)

    def training_step(self, batch, batch_idx):
        inputs, mask, labels = batch
        outputs = self.model(inputs, mask)
        loss = self.criterion(outputs, labels)
        self.log('train_loss', loss, on_step=False, on_epoch=True, prog_bar=True, sync_dist=True)
        return loss

    def validation_step(self, batch, batch_idx):
        inputs, mask, labels = batch
        outputs = self.model(inputs, mask)
        val_loss = self.criterion(outputs, labels)
        self.log('val_loss', val_loss, on_step=False, on_epoch=True, prog_bar=True, sync_dist=True)
        return val_loss

    def configure_optimizers(self):
        optimizer = torch.optim.Adam(self.parameters(), lr=0.001, fused=True)
        scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=100, gamma=0.1)
        return {'optimizer': optimizer, 'lr_scheduler': {'scheduler': scheduler, 'interval': 'epoch', 'frequency': 1}}
path = WEIGHTED_HAILRU_CHECKPOINT
checkpoint = torch.load(path, map_location='cpu', weights_only=False)
hailru = LightningModel(0.1, 10)
hailru.load_state_dict(checkpoint['state_dict'], strict=True)
hailru.eval()
hailru.to('cuda' if torch.cuda.is_available() else 'cpu')


In [ ]:
import math

class RMSNorm(nn.Module):

    def __init__(self, dim, eps=1e-08):
        super().__init__()
        self.eps = eps
        self.weight = nn.Parameter(torch.ones(dim))

    def forward(self, x):
        normed = x * torch.rsqrt(x.pow(2).mean(-1, keepdim=True) + self.eps)
        return normed * self.weight

class SwiGLU_FFN(nn.Module):

    def __init__(self, d_model):
        super().__init__()
        hidden_dim = int(d_model * 8 / 3)
        self.w_g = nn.Linear(d_model, hidden_dim, bias=False)
        self.w_1 = nn.Linear(d_model, hidden_dim, bias=False)
        self.w_2 = nn.Linear(hidden_dim, d_model, bias=False)

    def forward(self, x):
        return self.w_2(F.silu(self.w_g(x)) * self.w_1(x))

class WMHDA_WeightGenerator(nn.Module):

    def __init__(self, d_model, h):
        super().__init__()
        hidden_dim = d_model // 2
        self.w1 = nn.Linear(d_model, hidden_dim)
        self.w2 = nn.Linear(hidden_dim, h)

    def forward(self, x):
        x_bar = x.mean(dim=1)
        s = self.w2(F.gelu(self.w1(x_bar)))
        alpha = F.softmax(s, dim=-1)
        return alpha

class WMHDA_DIFFTransformerBlock(nn.Module):

    def __init__(self, d_model, h, layer_idx):
        super().__init__()
        self.d_model = d_model
        self.h = h
        self.d = d_model // (2 * h)
        self.ln1 = RMSNorm(d_model)
        self.q_proj = nn.Linear(d_model, h * 2 * self.d, bias=False)
        self.k_proj = nn.Linear(d_model, h * 2 * self.d, bias=False)
        self.v_proj = nn.Linear(d_model, h * 2 * self.d, bias=False)
        self.weight_gen = WMHDA_WeightGenerator(d_model, h)
        self.lambda_init = 0.8 - 0.6 * math.exp(-0.3 * (layer_idx - 1))
        self.lambda_q1 = nn.Parameter(torch.randn(self.d))
        self.lambda_k1 = nn.Parameter(torch.randn(self.d))
        self.lambda_q2 = nn.Parameter(torch.randn(self.d))
        self.lambda_k2 = nn.Parameter(torch.randn(self.d))
        self.rmsn = RMSNorm(2 * self.d)
        self.out_proj = nn.Linear(2 * self.d, d_model, bias=False)
        self.ln2 = RMSNorm(d_model)
        self.swiglu = SwiGLU_FFN(d_model)

    def forward(self, x):
        B, N, D = x.shape
        residual = x
        x_norm = self.ln1(x)
        alpha = self.weight_gen(x_norm)
        q = self.q_proj(x_norm).view(B, N, self.h, 2, self.d).permute(0, 2, 3, 1, 4)
        k = self.k_proj(x_norm).view(B, N, self.h, 2, self.d).permute(0, 2, 3, 1, 4)
        v = self.v_proj(x_norm).view(B, N, self.h, 2 * self.d).transpose(1, 2)
        q1, q2 = (q[:, :, 0], q[:, :, 1])
        k1, k2 = (k[:, :, 0], k[:, :, 1])
        lambda_val = torch.exp(torch.dot(self.lambda_q1, self.lambda_k1)) - torch.exp(torch.dot(self.lambda_q2, self.lambda_k2)) + self.lambda_init
        scale = 1.0 / math.sqrt(self.d)
        attn1 = F.softmax(q1 @ k1.transpose(-2, -1) * scale, dim=-1)
        attn2 = F.softmax(q2 @ k2.transpose(-2, -1) * scale, dim=-1)
        diff_attn = attn1 - lambda_val * attn2
        head_out = diff_attn @ v
        head_out_normed = self.rmsn(head_out)
        H_bar = head_out_normed * (1.0 - self.lambda_init)
        alpha = alpha.unsqueeze(-1).unsqueeze(-1)
        H_sum = (alpha * H_bar).sum(dim=1)
        attn_out = self.out_proj(H_sum)
        x = residual + attn_out
        x = x + self.swiglu(self.ln2(x))
        return x

class DoubleConv(nn.Module):

    def __init__(self, in_c, out_c):
        super().__init__()
        self.net = nn.Sequential(nn.Conv2d(in_c, out_c, kernel_size=3, padding=1), nn.BatchNorm2d(out_c), nn.ReLU(inplace=True), nn.Conv2d(out_c, out_c, kernel_size=3, padding=1), nn.BatchNorm2d(out_c), nn.ReLU(inplace=True))

    def forward(self, x):
        return self.net(x)

class DeconvUp(nn.Module):

    def __init__(self, in_c, out_c):
        super().__init__()
        self.up = nn.ConvTranspose2d(in_c, out_c, kernel_size=2, stride=2)

    def forward(self, x):
        return self.up(x)

class HailPre(nn.Module):

    def __init__(self, in_channels=102, num_frames_out=20, d_model=256, h=4, patch_size=16):
        super().__init__()
        self.patch_size = patch_size
        self.d_model = d_model
        self.patch_embed = nn.Conv2d(in_channels, d_model, kernel_size=patch_size, stride=patch_size)
        self.transformer_layers = nn.ModuleList([WMHDA_DIFFTransformerBlock(d_model=d_model, h=h, layer_idx=i + 1) for i in range(12)])
        self.deconv_z12 = DeconvUp(d_model, d_model // 2)
        self.conv_z9 = DoubleConv(d_model, d_model // 2)
        self.up_9_to_8 = DeconvUp(d_model // 2, d_model // 4)
        self.deconv_z6_stage1 = DeconvUp(d_model, d_model // 2)
        self.conv_z6 = DoubleConv(d_model // 2, d_model // 4)
        self.up_6_to_4 = DeconvUp(d_model // 4, d_model // 8)
        self.deconv_z3_stage1 = DeconvUp(d_model, d_model // 2)
        self.conv_z3_stage1 = DoubleConv(d_model // 2, d_model // 2)
        self.deconv_z3_stage2 = DeconvUp(d_model // 2, d_model // 4)
        self.conv_z3 = DoubleConv(d_model // 4, d_model // 8)
        self.up_3_to_2 = DeconvUp(d_model // 8, d_model // 16)
        self.conv_in = DoubleConv(in_channels, d_model // 16)
        self.decoder_stage1 = DoubleConv(d_model, d_model // 2)
        self.decoder_stage2 = DoubleConv(d_model // 2, d_model // 4)
        self.decoder_stage3 = DoubleConv(d_model // 4, d_model // 8)
        self.decoder_stage4 = DoubleConv(d_model // 8, d_model // 16)
        self.final_conv = nn.Conv2d(d_model // 16, num_frames_out, kernel_size=1)

    def forward(self, x):
        B, C_in, H_ori, W_ori = x.shape
        P = self.patch_size
        pad_h = (P - H_ori % P) % P
        pad_w = (P - W_ori % P) % P
        if pad_h > 0 or pad_w > 0:
            x_padded = F.pad(x, (0, pad_w, 0, pad_h))
        else:
            x_padded = x
        B, _, H, W = x_padded.shape
        feat_in = self.conv_in(x_padded)
        patches = self.patch_embed(x_padded)
        seq = patches.flatten(2).transpose(1, 2)
        N = seq.shape[1]
        device = seq.device
        pos = torch.arange(N, device=device, dtype=torch.float32).unsqueeze(1)
        dim = torch.arange(self.d_model, device=device, dtype=torch.float32).unsqueeze(0)
        pos_embed = torch.sin(pos / 10000 ** (2 * (dim // 2) / self.d_model))
        seq = seq + pos_embed.unsqueeze(0)
        features = {}
        for i, layer in enumerate(self.transformer_layers):
            seq = layer(seq)
            if i + 1 in [3, 6, 9, 12]:
                features[f'Z{i + 1}'] = seq.transpose(1, 2).view(B, -1, H // P, W // P)
        Z3, Z6, Z9, Z12 = (features['Z3'], features['Z6'], features['Z9'], features['Z12'])
        z12_up = self.deconv_z12(Z12)
        z9_conv = self.conv_z9(Z9)
        z9_up = F.interpolate(z9_conv, scale_factor=2, mode='bilinear', align_corners=False)
        d1 = self.decoder_stage1(torch.cat([z12_up, z9_up], dim=1))
        d1_up = self.up_9_to_8(d1)
        z6_up = self.deconv_z6_stage1(Z6)
        z6_up = F.interpolate(z6_up, scale_factor=2, mode='bilinear', align_corners=False)
        z6_conv = self.conv_z6(z6_up)
        d2 = self.decoder_stage2(torch.cat([d1_up, z6_conv], dim=1))
        d2_up = self.up_6_to_4(d2)
        z3_up = self.deconv_z3_stage2(self.conv_z3_stage1(self.deconv_z3_stage1(Z3)))
        z3_up = F.interpolate(z3_up, scale_factor=2, mode='bilinear', align_corners=False)
        z3_conv = self.conv_z3(z3_up)
        d3 = self.decoder_stage3(torch.cat([d2_up, z3_conv], dim=1))
        d3_up = self.up_3_to_2(d3)
        d4 = self.decoder_stage4(torch.cat([d3_up, feat_in], dim=1))
        out_padded = self.final_conv(d4)
        out = out_padded[:, :, :H_ori, :W_ori]
        return out

class DIFFLightningModel(pl.LightningModule):

    def __init__(self):
        super().__init__()
        self.model = HailPre(in_channels=102, num_frames_out=20, d_model=128, h=4)

    def forward(self, x):
        return self.model(x)
checkpoint = torch.load(DIFF_T_CHECKPOINT, map_location='cpu', weights_only=False)
difft = DIFFLightningModel()
difft.load_state_dict(checkpoint['state_dict'], strict=True)
difft.eval()
difft.to('cuda' if torch.cuda.is_available() else 'cpu')


In [ ]:
class TDconv2d_layer(nn.Module):

    def __init__(self, inchannels):
        super(TDconv2d_layer, self).__init__()
        self.c = inchannels
        self.act = nn.LeakyReLU(inplace=True)
        self.tdconv = nn.Sequential(nn.Conv2d(self.c, self.c, kernel_size=3, stride=1, padding=1, bias=False), nn.BatchNorm2d(self.c), self.act, nn.Conv2d(self.c, self.c, kernel_size=3, stride=1, padding=1, bias=False), nn.BatchNorm2d(self.c))

    def forward(self, x):
        identity = x
        out = self.tdconv(x)
        out = 0.3 * out + identity
        return self.act(out)

class HRUnet2D(nn.Module):

    def __init__(self, in_channels, out_channels):
        super(HRUnet2D, self).__init__()
        chans = [128, 256, 512]
        ks = 3
        self.act = nn.LeakyReLU(inplace=True)
        self.act_output = nn.LeakyReLU(inplace=True)
        self.start_conv = nn.Sequential(nn.Conv2d(in_channels, chans[0], kernel_size=3, stride=1, padding=1, bias=False), nn.BatchNorm2d(chans[0]), self.act)
        self.ResBlock_11 = TDconv2d_layer(chans[0])
        self.ResBlock_12 = TDconv2d_layer(chans[0])
        self.DownConv_1 = nn.Sequential(nn.MaxPool2d(kernel_size=2, stride=2), nn.Conv2d(chans[0], chans[1], kernel_size=1, stride=1, bias=False), nn.BatchNorm2d(chans[1]), self.act)
        self.ResBlock_21 = TDconv2d_layer(chans[1])
        self.ResBlock_22 = TDconv2d_layer(chans[1])
        self.DownConv_2 = nn.Sequential(nn.MaxPool2d(kernel_size=2, stride=2), nn.Conv2d(chans[1], chans[2], kernel_size=1, stride=1, bias=False), nn.BatchNorm2d(chans[2]), self.act)
        self.ResBlock_31 = TDconv2d_layer(chans[2])
        self.ResBlock_32 = TDconv2d_layer(chans[2])
        self.ResBlock_33 = TDconv2d_layer(chans[2])
        self.ResBlock_34 = TDconv2d_layer(chans[2])
        self.UpConv_1 = nn.ConvTranspose2d(chans[2], chans[1], kernel_size=2, stride=2)
        self.UpConv_next_1 = nn.Sequential(nn.Conv2d(chans[1] * 2, chans[1], kernel_size=ks, stride=1, padding=1, bias=False), nn.BatchNorm2d(chans[1]), self.act)
        self.ResBlock_23 = TDconv2d_layer(chans[1])
        self.ResBlock_24 = TDconv2d_layer(chans[1])
        self.UpConv_2 = nn.ConvTranspose2d(chans[1], chans[0], kernel_size=2, stride=2)
        self.UpConv_next_2 = nn.Sequential(nn.Conv2d(chans[0] * 2, chans[0], kernel_size=ks, stride=1, padding=1, bias=False), nn.BatchNorm2d(chans[0]), self.act)
        self.ResBlock_13 = TDconv2d_layer(chans[0])
        self.ResBlock_14 = TDconv2d_layer(chans[0])
        self.final_conv = nn.Sequential(nn.Conv2d(chans[0], out_channels, kernel_size=3, stride=1, padding=1), self.act_output)

    def _pad_to_match(self, x, skip_x):
        diffY = skip_x.size()[2] - x.size()[2]
        diffX = skip_x.size()[3] - x.size()[3]
        return F.pad(x, [diffX // 2, diffX - diffX // 2, diffY // 2, diffY - diffY // 2])

    def forward(self, x, mask=None):
        x = self.start_conv(x)
        x = self.ResBlock_11(x)
        x = self.ResBlock_12(x)
        x_pass_1 = x
        x = self.DownConv_1(x)
        x = self.ResBlock_21(x)
        x = self.ResBlock_22(x)
        x_pass_2 = x
        x = self.DownConv_2(x)
        x = self.ResBlock_31(x)
        x = self.ResBlock_32(x)
        x = self.ResBlock_33(x)
        x = self.ResBlock_34(x)
        x = self.UpConv_1(x)
        x = self._pad_to_match(x, x_pass_2)
        x = torch.cat((x, x_pass_2), dim=1)
        x = self.UpConv_next_1(x)
        x = self.ResBlock_23(x)
        x = self.ResBlock_24(x)
        x = self.UpConv_2(x)
        x = self._pad_to_match(x, x_pass_1)
        x = torch.cat((x, x_pass_1), dim=1)
        x = self.UpConv_next_2(x)
        x = self.ResBlock_13(x)
        x = self.ResBlock_14(x)
        x = self.final_conv(x)
        if mask is not None:
            x = x * mask
        return x

class Loss(nn.Module):

    def __init__(self, alpha, beta):
        super(Loss, self).__init__()
        self.alpha = alpha
        self.beta = beta

    def forward(self, y_pred, y_true):
        base_temp = torch.square(y_pred - y_true) * ((y_true + self.alpha) / (1 + self.alpha))
        if torch.sum(y_true) > 0:
            temp = base_temp
        else:
            temp = self.beta * base_temp
        divisor = temp.shape[1] * temp.shape[2] * temp.shape[3]
        return torch.sum(temp) / divisor

class LightningModel(pl.LightningModule):

    def __init__(self, alpha, beta):
        super().__init__()
        self.model = HRUnet2D(in_channels=102, out_channels=20)
        self.criterion = Loss(alpha, beta)

    def forward(self, x, mask):
        return self.model(x, mask)

    def training_step(self, batch, batch_idx):
        inputs, mask, labels = batch
        outputs = self.model(inputs, mask)
        loss = self.criterion(outputs, labels)
        self.log('train_loss', loss, on_step=False, on_epoch=True, prog_bar=True, sync_dist=True)
        return loss

    def validation_step(self, batch, batch_idx):
        inputs, mask, labels = batch
        outputs = self.model(inputs, mask)
        val_loss = self.criterion(outputs, labels)
        self.log('val_loss', val_loss, on_step=False, on_epoch=True, prog_bar=True, sync_dist=True)
        return val_loss

    def configure_optimizers(self):
        optimizer = torch.optim.Adam(self.parameters(), lr=0.0001, fused=True)
        scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=50, gamma=0.5)
        return {'optimizer': optimizer, 'lr_scheduler': {'scheduler': scheduler, 'interval': 'epoch', 'frequency': 1}}
path = UNET_CHECKPOINT
checkpoint = torch.load(path, map_location='cpu', weights_only=False)
tdunet = LightningModel(0.1, 10)
tdunet.load_state_dict(checkpoint['state_dict'], strict=True)
tdunet.eval()
tdunet.to('cuda' if torch.cuda.is_available() else 'cpu')


In [ ]:
from collections.abc import Mapping
from datetime import datetime
from pathlib import Path
import json
import math
import re
import warnings
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.collections import LineCollection
from matplotlib.colors import BoundaryNorm, LinearSegmentedColormap, Normalize
from matplotlib.lines import Line2D
from matplotlib.cm import ScalarMappable

def _as_3d(value, name):
    if hasattr(value, 'detach'):
        value = value.detach().float().cpu().numpy()
    arr = np.asarray(value)
    if arr.ndim != 3:
        raise ValueError(f'{name} must have shape (T, H, W); received {arr.shape}。')
    return arr

def _resolve_start_i(sample_path=None, start_i=None, case_id=None):
    if start_i is not None:
        if isinstance(start_i, (str, bool)) or int(start_i) != start_i or start_i < 0:
            raise ValueError('start_i must be a nonnegative integer.')
        start_i = int(start_i)
    if sample_path is not None:
        parts = str(sample_path).replace('\\', '/').split('/')
        found = re.fullmatch('(\\d+)\\.npz', parts[-1], flags=re.IGNORECASE)
        if found is None:
            raise ValueError('sample_path must use the numeric sample filename, such as output/3.npz.')
        path_index = int(found.group(1))
        case_parts = [part for part in parts if re.fullmatch('HA_\\d+_\\d{8}_\\d+', part)]
        if case_id is not None and case_parts and (case_id not in case_parts):
            raise ValueError(f'sample_path does not match case_id={case_id!r} does not match.')
        if start_i is not None and start_i != path_index:
            raise ValueError(f'start_i={start_i} differs from sample filename {parts[-1]} does not match.')
        start_i = path_index
    if start_i is None:
        raise ValueError('Supply sample_path or start_i; a sample-list position does not identify its original time window.')
    return start_i

def _cr_file_time(file_path, reference_time):
    basename = str(file_path).replace('\\', '/').rsplit('/', 1)[-1]
    found = re.search('_(\\d{4})-(\\d{4}(?:\\d{2})?)(?=_|\\.)', basename)
    if found is None:
        raise ValueError(f'Cannot parse CR filename timestamp (expected MMDD-HHMM): {basename}')
    month_day, hour_minute = found.groups()
    candidates = []
    for year in (reference_time.year - 1, reference_time.year, reference_time.year + 1):
        try:
            candidates.append(datetime(year, int(month_day[:2]), int(month_day[2:]), int(hour_minute[:2]), int(hour_minute[2:4]), int(hour_minute[4:]) if len(hour_minute) == 6 else 0))
        except ValueError:
            continue
    if not candidates:
        raise ValueError(f'Invalid time in CR filename: {basename}')
    return min(candidates, key=lambda dt: abs(dt - reference_time))

def load_cr_from_data_dic(data_dic, case_id, *, sample_path=None, start_i=None, input_len=10, n_frames=20, time_step=6, spatial_shape=None, print_matches=True):
    if not isinstance(data_dic, Mapping) or case_id not in data_dic:
        raise KeyError(f'data_dic has no case_id={case_id!r}。')
    files = data_dic[case_id].get('CR')
    if files is None or isinstance(files, (str, Path)) or len(files) == 0:
        raise ValueError(f"data_dic[{case_id!r}]['CR'] must be a nonempty list of CR file paths.")
    if int(n_frames) != n_frames or n_frames < 1:
        raise ValueError('n_frames must be a positive integer.')
    if int(input_len) != input_len or input_len < 1:
        raise ValueError('input_len must be a positive integer.')
    if not np.isfinite(time_step) or time_step <= 0:
        raise ValueError('time_step must be positive.')
    start_i = _resolve_start_i(sample_path, start_i, case_id)
    output_start = start_i + int(input_len)
    output_stop = output_start + int(n_frames)
    if output_stop > len(files):
        raise ValueError(f'Window exceeds the sequence: start_i={start_i}, input_len={input_len}, output_len={n_frames}; required final CR index {output_stop - 1}, but only {len(files)} frames.')
    input_last_index = output_start - 1
    input_last_time = _cr_file_time(files[input_last_index], datetime(2000, 7, 1))
    previous_time = input_last_time
    matches = []
    for frame_index, source_index in enumerate(range(output_start, output_stop)):
        file_path = str(files[source_index])
        radar_time = _cr_file_time(file_path, previous_time)
        if radar_time <= previous_time:
            raise ValueError(f'CR source index {source_index} is not chronological; check the original sequence. CR is not reordered independently of labels.')
        matches.append({'frame_index': frame_index, 'cr_time': radar_time, 'lead_minutes': (radar_time - input_last_time).total_seconds() / 60, 'nominal_lead_minutes': (frame_index + 1) * float(time_step), 'source_index': source_index, 'start_i': start_i, 'input_last_source_index': input_last_index, 'input_last_time': input_last_time, 'year_is_placeholder': True, 'path': file_path})
        previous_time = radar_time
    loaded, frames = ({}, [])
    expected_shape = tuple(spatial_shape) if spatial_shape is not None else None
    for match in matches:
        file_path = match['path']
        if file_path not in loaded:
            try:
                data = np.load(file_path, allow_pickle=False)
            except FileNotFoundError as exc:
                raise FileNotFoundError(f'Matched CR timestamp, but the file is missing: {file_path}') from exc
            if isinstance(data, np.lib.npyio.NpzFile):
                data.close()
                raise ValueError('CR entries must point to numeric .npy arrays.')
            arr = np.asarray(data).squeeze()
            if arr.ndim != 2 or not np.issubdtype(arr.dtype, np.number) or np.iscomplexobj(arr):
                raise ValueError(f'CR must be a two-dimensional real array; received {arr.shape}/{arr.dtype}：{file_path}')
            if expected_shape is None:
                expected_shape = arr.shape
            if arr.shape != expected_shape:
                raise ValueError(f'CR grid {arr.shape} differs from the expected shape {expected_shape} does not match: {file_path}. Use the same crop and grid as output_data.')
            loaded[file_path] = arr.astype(np.float32, copy=False)
        frames.append(loaded[file_path])
    cr_dbz = np.stack(frames, axis=0)
    if print_matches:
        print(f'Case: {case_id}; start_i={start_i}; input indices {start_i}..{input_last_index}; output indices {output_start}..{output_stop - 1}')
        print(f'Last input CR: {input_last_time:%m-%d %H:%M}; filename dates have no year.')
        print('output frame | raw CR / label index | valid time  | actual lead/min')
        for match in matches:
            print(f"{match['frame_index']:12d} | {match['source_index']:20d} | {match['cr_time']:%m-%d %H:%M} | {match['lead_minutes']:g}")
    return (cr_dbz, matches)

def _station_obs_from_raw_labels(data_dic, case_id, matches, case_points, shape):
    labels = data_dic[case_id].get('label')
    if labels is None or len(labels) != len(data_dic[case_id]['CR']):
        raise ValueError("data_dic[case_id]['label'] and CR must have equal lengths.")
    height, width = shape
    records = list(case_points.items())
    by_coord = {}
    observations = np.full((len(matches), len(records)), np.nan)
    for col, (_, coords) in enumerate(records):
        try:
            xy = np.asarray(coords[:2], dtype=float)
            if xy.shape != (2,) or not np.isfinite(xy).all():
                continue
            if not np.allclose(xy, np.rint(xy), atol=1e-06, rtol=0):
                continue
            x, y = np.rint(xy).astype(int)
        except (ValueError, TypeError, IndexError):
            continue
        if 0 <= x < width and 0 <= y < height:
            by_coord.setdefault((x, y), []).append(col)
            observations[:, col] = 0
    for frame, match in enumerate(matches):
        points = labels[match['source_index']]
        if points is None:
            observations[frame] = np.nan
            continue
        for point in points:
            xy = np.asarray(point[:2], dtype=float)
            if xy.shape != (2,) or not np.isfinite(xy).all() or (not np.allclose(xy, np.rint(xy), atol=1e-06, rtol=0)):
                raise ValueError(f"Raw label[{match['source_index']}] contains invalid grid coordinates: {point}")
            coord = tuple(np.rint(xy).astype(int))
            if coord not in by_coord:
                raise ValueError(f"Raw label[{match['source_index']}] contains hail coordinates {coord} is absent from the station dictionary; use the matching mask_dic_all and check grid bounds.")
            observations[frame, by_coord[coord]] = 1
    return observations

def _geometry_lines(geometry):
    if geometry is None:
        return
    kind = geometry.get('type')
    coords = geometry.get('coordinates', [])
    if kind == 'LineString':
        yield coords
    elif kind in ('MultiLineString', 'Polygon'):
        yield from coords
    elif kind == 'MultiPolygon':
        for polygon in coords:
            yield from polygon
    elif kind == 'GeometryCollection':
        for child in geometry.get('geometries', []):
            yield from _geometry_lines(child)

def _load_city_lines(city_boundary_path, geo_extent):
    if city_boundary_path is None:
        bundled_name = 'china_prefecture_WGS84.geojson'
        script_path = globals().get('__file__')
        candidates = []
        if script_path:
            candidates.append(Path(script_path).resolve().parent / bundled_name)
        candidates.append(Path.cwd() / bundled_name)
        path = next((candidate for candidate in candidates if candidate.is_file()), None)
        if path is None:
            raise FileNotFoundError(f'Boundary file not found: {bundled_name}. Supply a WGS84 boundary file using city_boundary_path; boundary data are not bundled.')
    else:
        path = Path(city_boundary_path).expanduser()
    if not path.is_file():
        raise FileNotFoundError(f'Boundary file not found: {path}. Set CITY_BOUNDARY_PATH in project_config.py to an existing boundary file.')
    if path.suffix.lower() in ('.json', '.geojson'):
        with path.open(encoding='utf-8-sig') as handle:
            obj = json.load(handle)
        crs_name = str((obj.get('crs') or {}).get('properties', {}).get('name', ''))
        if crs_name and (not (crs_name.endswith('4326') or 'CRS84' in crs_name.upper())):
            raise ValueError('GeoJSON must use WGS84 longitude and latitude; reproject to EPSG:4326.')
        if obj.get('type') == 'FeatureCollection':
            geometries = [feature.get('geometry') for feature in obj['features']]
        elif obj.get('type') == 'Feature':
            geometries = [obj.get('geometry')]
        else:
            geometries = [obj]
    else:
        try:
            import geopandas as gpd
        except ImportError as exc:
            raise ImportError('Reading non-GeoJSON boundary files requires geopandas.') from exc
        cities = gpd.read_file(path)
        if cities.crs is None:
            raise ValueError('Boundary CRS is missing; establish its CRS before transforming to EPSG:4326.')
        cities = cities.to_crs(epsg=4326)
        geometries = [geom.__geo_interface__ for geom in cities.geometry if geom is not None and (not geom.is_empty)]
    west, east, south, north = geo_extent
    lines = []
    for geometry in geometries:
        for coords in _geometry_lines(geometry):
            arr = np.asarray(coords, dtype=float)
            if arr.ndim != 2 or len(arr) < 2 or arr.shape[1] < 2:
                continue
            arr = arr[:, :2]
            if not np.isfinite(arr).all():
                continue
            if arr[:, 0].max() < west or arr[:, 0].min() > east or arr[:, 1].max() < south or (arr[:, 1].min() > north):
                continue
            lines.append(arr)
    if not lines:
        raise ValueError('No boundary intersects geo_extent; check geometry, extent, and CRS.')
    return lines

def _station_arrays(case_points, shape, label, station_obs, label_thre):
    nt, height, width = shape
    records = list(case_points.items())
    keys, indices, xs, ys = ([], [], [], [])
    for original_index, (key, coords) in enumerate(records):
        try:
            xy = np.asarray(coords[:2], dtype=float)
            if len(xy) != 2 or not np.isfinite(xy).all():
                continue
            if not np.allclose(xy, np.rint(xy), atol=1e-06, rtol=0):
                continue
            x, y = np.rint(xy).astype(int)
        except (ValueError, TypeError, IndexError):
            continue
        if 0 <= x < width and 0 <= y < height:
            keys.append(key)
            indices.append(original_index)
            xs.append(x)
            ys.append(y)
    if not keys:
        raise ValueError('No valid stations; check the (x, y) coordinates in mask_dic.')
    if len(keys) < len(records):
        warnings.warn(f'Skipped {len(records) - len(keys)} stations with invalid or out-of-bounds coordinates.', stacklevel=2)
    xs, ys = (np.asarray(xs), np.asarray(ys))
    if station_obs is None:
        values = label[:, ys, xs]
        obs_valid = np.isfinite(values)
        obs_yes = values > label_thre
        warnings.warn('station_obs was not provided; station labels are thresholded. For diffused labels, supply original binary station observations.', stacklevel=2)
    else:
        if isinstance(station_obs, Mapping):
            columns = []
            for key in keys:
                column = np.asarray(station_obs.get(key, np.full(nt, np.nan)), dtype=float)
                if column.shape != (nt,):
                    raise ValueError(f'station_obs[{key!r}] must be a sequence of length {nt} with values 0, 1, or NaN.')
                columns.append(column)
            values = np.column_stack(columns)
        else:
            values = np.asarray(station_obs, dtype=float)
            if values.shape != (nt, len(records)):
                raise ValueError(f'station_obs must have shape {(nt, len(records))}; columns must follow list(mask_dic[case_id].keys()).')
            values = values[:, indices]
        if np.isinf(values).any() or not np.isin(values[np.isfinite(values)], [0, 1]).all():
            raise ValueError('station_obs only accepts 0, 1, and NaN; encode missing observations as NaN.')
        obs_valid = np.isfinite(values)
        obs_yes = values == 1
    return (xs, ys, obs_yes, obs_valid)

def plot_model_comparison(data_dict, mask_dic, thre=0.5, case_id='HA_21_06041749_00046', title='Case_ID : HA_21_06041749_00046', save=False, *, cr_dbz=None, data_dic=None, sample_path=None, start_i=None, input_len=10, print_cr_matches=True, annotate_cr_time=False, geo_extent=None, sta_dic=None, city_boundary_path=None, station_obs=None, display_interval=12, time_step=6, time_label_mode='display', first_lead_min=None, time_indices=None, include_last=False, cols_per_block=5, stats_scope='all', label_thre=0.5, show_probability=True, probability_min=0.02, origin='upper', panel_inches=2.7, font_scale=1.2, figsize=None, save_dir=OUTPUT_ROOT, dpi=600, show=True):
    names = list(data_dict)
    if len(names) < 2:
        raise ValueError('data_dict must contain Label and at least one model.')
    if 'Label' in names and names[0] != 'Label':
        raise ValueError('Label must be the first entry in data_dict.')
    arrays = {name: _as_3d(data_dict[name], name) for name in names}
    label = arrays[names[0]]
    nt, height, width = label.shape
    if nt < 1 or min(height, width) < 2:
        raise ValueError('At least one frame and two grid points per spatial dimension are required.')
    for name, arr in arrays.items():
        if arr.shape != label.shape:
            raise ValueError(f'{name} has shape {arr.shape} differs from Label shape {label.shape} does not match.')
    if case_id not in mask_dic:
        raise KeyError(f'mask_dic has no case_id={case_id!r}。')
    cr_matches = None
    raw_station_records = False
    if cr_dbz is not None and data_dic is not None:
        raise ValueError('Supply either cr_dbz or data_dic, not both.')
    if cr_dbz is None:
        if data_dic is None:
            raise ValueError('Supply data_dic and sample_path (or start_i), or provide cr_dbz directly.')
        cr_dbz, cr_matches = load_cr_from_data_dic(data_dic, case_id, sample_path=sample_path, start_i=start_i, input_len=input_len, n_frames=nt, time_step=time_step, spatial_shape=(height, width), print_matches=print_cr_matches)
        if station_obs is None:
            station_obs = _station_obs_from_raw_labels(data_dic, case_id, cr_matches, mask_dic[case_id], (height, width))
            raw_station_records = True
    radar = _as_3d(cr_dbz, 'cr_dbz')
    if radar.shape != label.shape:
        raise ValueError(f'CR must match the Label time and spatial grid; expected {label.shape}; received {radar.shape}。')
    if geo_extent is None:
        station_id = case_id.split('_')[-1]
        if sta_dic is None or station_id not in sta_dic:
            raise ValueError('Supply sta_dic or geo_extent.')
        bounds = np.asarray(sta_dic[station_id], dtype=float)
        if bounds.shape != (4,):
            raise ValueError('sta_dic extents must be [lons, lone, lats, late].')
        geo_extent = (bounds[0], bounds[1], bounds[3], bounds[2])
    extent = np.asarray(geo_extent, dtype=float)
    if extent.shape != (4,) or not np.isfinite(extent).all():
        raise ValueError('geo_extent must be (lon_min, lon_max, lat_min, lat_max).')
    west, east, south, north = extent
    if not (-180 <= west < east <= 180 and -90 <= south < north <= 90):
        raise ValueError('Use ordered longitude/latitude extents (west, east, south, north).')
    if origin not in ('upper', 'lower'):
        raise ValueError("origin must be 'upper' or 'lower'.")
    if not (0 <= thre <= 1 and 0 <= label_thre <= 1 and (0 <= probability_min < 1)):
        raise ValueError('thre and label_thre must be in [0, 1]; probability_min must be in [0, 1).')
    if time_step <= 0 or cols_per_block < 1 or int(cols_per_block) != cols_per_block or (panel_inches <= 0):
        raise ValueError('time_step and panel_inches must be positive; cols_per_block must be a positive integer.')
    if not np.isfinite(font_scale) or font_scale <= 0:
        raise ValueError('font_scale must be positive.')
    if not np.isfinite(display_interval) or display_interval <= 0:
        raise ValueError('display_interval must be positive.')
    if time_indices is None:
        stride = display_interval / time_step
        if stride < 1 or not np.isclose(stride, round(stride)):
            raise ValueError('display_interval must be a positive integer multiple of time_step.')
        selected = np.arange(0, nt, int(round(stride)))
    else:
        selected = np.asarray(time_indices)
        if selected.ndim != 1 or selected.size == 0:
            raise ValueError('time_indices must be a nonempty one-dimensional sequence.')
        if not np.issubdtype(selected.dtype, np.integer):
            raise ValueError('time_indices must contain integer source-frame indices.')
        if np.any(selected < 0) or np.any(selected >= nt):
            raise ValueError(f'time_indices must be between 0 and {nt - 1}.')
        selected = np.unique(selected)
    if include_last:
        selected = np.unique(np.append(selected, nt - 1))
    if stats_scope not in ('all', 'displayed'):
        raise ValueError("stats_scope must be 'all' or 'displayed'.")
    stat_indices = np.arange(nt) if stats_scope == 'all' else selected
    if time_label_mode not in ('display', 'actual', 'nominal'):
        raise ValueError("time_label_mode must be 'display', 'actual', or 'nominal'.")
    actual_lead_minutes = np.asarray([match['lead_minutes'] for match in cr_matches]) if cr_matches is not None else None
    if time_label_mode == 'display':
        initial_lead = 0 if first_lead_min is None else first_lead_min
        lead_minutes = initial_lead + np.arange(nt) * time_step
        shown_minutes = {int(t): initial_lead + col_index * display_interval for col_index, t in enumerate(selected)}
        nominal_labels = False
    elif cr_matches is not None and time_label_mode == 'actual':
        if first_lead_min is not None:
            raise ValueError('Actual lead times use CR timestamps; do not also set first_lead_min.')
        lead_minutes = actual_lead_minutes
        shown_minutes = {int(t): lead_minutes[t] for t in selected}
        nominal_labels = False
    else:
        initial_lead = time_step if first_lead_min is None else first_lead_min
        lead_minutes = initial_lead + np.arange(nt) * time_step
        shown_minutes = {int(t): lead_minutes[t] for t in selected}
        nominal_labels = True
    city_lines = _load_city_lines(city_boundary_path, extent)
    xs, ys, obs_yes, obs_valid = _station_arrays(mask_dic[case_id], label.shape, label, station_obs, label_thre)
    hail_frames = np.flatnonzero(np.any(obs_yes & obs_valid, axis=1))
    if len(hail_frames) and (not np.intersect1d(selected, hail_frames).size):
        warnings.warn(f'Displayed frames contain no hail observations. Hail frames are {hail_frames.tolist()}; specify time_indices or set display_interval=6 to include them.', stacklevel=2)
    dx, dy = ((east - west) / (width - 1), (north - south) / (height - 1))
    station_lon = west + xs * dx
    station_lat = north - ys * dy if origin == 'upper' else south + ys * dy
    image_extent = (west - dx / 2, east + dx / 2, south - dy / 2, north + dy / 2)
    model_stats, forecast_yes = ({}, {})
    for name in names[1:]:
        values = arrays[name][:, ys, xs]
        forecast_yes[name] = np.isfinite(values) & (values > thre)
        valid = (obs_valid & np.isfinite(values))[stat_indices]
        truth = obs_yes[stat_indices]
        predicted = forecast_yes[name][stat_indices]
        model_stats[name] = {'Hit': int(np.count_nonzero(valid & truth & predicted)), 'Mis': int(np.count_nonzero(valid & truth & ~predicted)), 'FA': int(np.count_nonzero(valid & ~truth & predicted)), 'N_eval': int(np.count_nonzero(valid))}
    nrows = len(names)
    ncols = min(int(cols_per_block), len(selected))
    nblocks = math.ceil(len(selected) / ncols)
    left_in = 1.65
    right_in = 1.9 + (1.15 if nblocks == 1 else 0)
    top_in, bottom_in = (1.36, 0.25)
    col_gap, row_gap, block_gap = (0.08, 0.09, 0.58)
    panel_height = panel_inches * (north - south) / (east - west)
    block_height = nrows * panel_height + (nrows - 1) * row_gap
    layout_width = left_in + right_in + ncols * panel_inches + (ncols - 1) * col_gap
    layout_height = top_in + bottom_in + nblocks * block_height + (nblocks - 1) * block_gap
    fig = plt.figure(figsize=figsize or (layout_width, layout_height), facecolor='white')
    fig.cr_matches = cr_matches
    fig.station_obs = station_obs
    fig.lead_minutes = lead_minutes
    fig.actual_lead_minutes = actual_lead_minutes
    fig.display_minutes = np.asarray([shown_minutes[int(t)] for t in selected])
    fig.display_indices = selected.copy()
    fig.hail_frame_indices = hail_frames
    fig.suptitle(title or case_id, fontsize=20 * font_scale, fontweight='bold', y=1 - 0.12 / layout_height, va='top')
    red, black = ('#d92d20', '#202020')
    has_observations = station_obs is not None
    legend_handles = [Line2D([], [], marker='o', ls='', markerfacecolor=black, markeredgecolor='white', markersize=6, label='No hailfall recorded' if raw_station_records else 'No hailfall observed' if has_observations else 'No-hail label'), Line2D([], [], marker='o', linestyle='none', markerfacecolor=red, markeredgecolor=red, markeredgewidth=3, markersize=np.sqrt(50), label='Hailfall recorded'), Line2D([], [], marker='o', linestyle='none', markerfacecolor='none', markeredgecolor=red, markeredgewidth=2, markersize=np.sqrt(70), label='Forecast hailfall'), Line2D([], [], marker='^', linestyle='none', markerfacecolor='none', markeredgecolor=red, markeredgewidth=2, markersize=np.sqrt(70), label='Missed hailfall')]
    fig.legend(handles=legend_handles, loc='upper center', ncol=4, bbox_to_anchor=(0.5, 1 - 0.4 / layout_height), frameon=False, prop={'size': 16, 'weight': 'semibold'}, columnspacing=2, handletextpad=0.45)
    cr_levels = np.arange(15, 75, 5)
    cr_cmap = LinearSegmentedColormap.from_list('cr_teal', ['#edf9f5', '#b5e2d9', '#69c6b7', '#2e9f92', '#14796e'], N=len(cr_levels) - 1)
    cr_cmap.set_bad((0, 0, 0, 0))
    cr_norm = BoundaryNorm(cr_levels, cr_cmap.N, clip=True)
    prob_cmap = plt.get_cmap('Purples').copy()
    prob_cmap.set_bad((0, 0, 0, 0))
    prob_norm = Normalize(0, 1)
    axes = {}
    for block in range(nblocks):
        times = selected[block * ncols:(block + 1) * ncols]
        block_top = layout_height - top_in - block * (block_height + block_gap)
        for row, name in enumerate(names):
            for col, t in enumerate(times):
                ax = fig.add_axes([(left_in + col * (panel_inches + col_gap)) / layout_width, (block_top - row * (panel_height + row_gap) - panel_height) / layout_height, panel_inches / layout_width, panel_height / layout_height])
                axes[block, row, col] = ax
                ax.set_label(f'{name}|{shown_minutes[int(t)]:g}min')
                ax.set_facecolor('#fcfcfc')
                ax.add_collection(LineCollection(city_lines, colors='#aeb4ba', linewidths=0.55, zorder=1))
                if row == 0:
                    field = np.ma.masked_where(~np.isfinite(radar[t]) | (radar[t] < 15), radar[t])
                    ax.imshow(field, origin=origin, extent=image_extent, interpolation='nearest', cmap=cr_cmap, norm=cr_norm, alpha=0.82, zorder=2)
                    no_hail = obs_valid[t] & ~obs_yes[t]
                    hail = obs_valid[t] & obs_yes[t]
                    ax.scatter(station_lon[no_hail], station_lat[no_hail], s=16, marker='o', c=black, edgecolors='white', linewidths=0.35, zorder=4, clip_on=True)
                    ax.scatter(station_lon[hail], station_lat[hail], s=50, marker='o', facecolors=red, edgecolors=red, linewidths=3, zorder=5, clip_on=True)
                    if annotate_cr_time and cr_matches is not None:
                        matched = cr_matches[t]
                        text = f"CR {matched['cr_time']:%m-%d %H:%M}"
                        ax.text(0.02, 0.02, text, transform=ax.transAxes, fontsize=8 * font_scale, color='#44515b', ha='left', va='bottom', zorder=6, bbox=dict(facecolor='white', edgecolor='none', alpha=0.82, pad=1.4))
                else:
                    if show_probability:
                        field = np.ma.masked_where(~np.isfinite(arrays[name][t]) | (arrays[name][t] < probability_min), arrays[name][t])
                        ax.imshow(field, origin=origin, extent=image_extent, interpolation='nearest', cmap=prob_cmap, norm=prob_norm, alpha=0.84, zorder=2)
                    predicted = forecast_yes[name][t]
                    ax.scatter(station_lon[predicted], station_lat[predicted], s=70, marker='o', facecolors='none', edgecolors=red, linewidths=2, zorder=5, clip_on=True)
                    missed = obs_valid[t] & obs_yes[t] & np.isfinite(arrays[name][t, ys, xs]) & ~predicted
                    ax.scatter(station_lon[missed], station_lat[missed], s=70, marker='^', facecolors='none', edgecolors=red, linewidths=2, zorder=6, clip_on=True)
                ax.set_xlim(image_extent[:2])
                ax.set_ylim(image_extent[2:])
                ax.set_aspect('equal', adjustable='box')
                ax.set_xticks([])
                ax.set_yticks([])
                for spine in ax.spines.values():
                    spine.set_color('#d9dde1')
                    spine.set_linewidth(0.5)
                if col == 0:
                    ax.set_ylabel(name, rotation=0, ha='right', va='center', labelpad=15, fontsize=14 * font_scale, fontweight='bold')
                if row == 0:
                    nominal_text = ' (nominal)' if nominal_labels else ''
                    display_time = shown_minutes[int(t)]
                    ax.set_title(f'{display_time:g}min', fontsize=16, pad=10)
    stat_offset = 1.94 if nblocks == 1 else 0.2
    stat_x = (layout_width - right_in + stat_offset) / layout_width
    last_block = nblocks - 1
    for row, name in enumerate(names[1:], start=1):
        pos = axes[last_block, row, 0].get_position()
        stats = model_stats[name]
        fig.text(stat_x, (pos.y0 + pos.y1) / 2, f"Hit: {stats['Hit']}\nMis: {stats['Mis']}\nFA:  {stats['FA']}", fontsize=18, va='center', ha='left', color='#26313e', linespacing=1.6)
    grid_right = axes[0, 0, ncols - 1].get_position().x1
    bar_left = grid_right + 0.27 / layout_width
    bar_width = 0.23 / layout_width

    def colorbar_rect(start_row, end_row):
        top = axes[0, start_row, ncols - 1].get_position().y1
        bottom = axes[0, end_row, ncols - 1].get_position().y0
        inset = 0.075 * (top - bottom)
        return [bar_left, bottom + inset, bar_width, top - bottom - 2 * inset]
    cr_last_row = 1 if nrows >= 3 else 0
    cr_bar = fig.add_axes(colorbar_rect(0, cr_last_row), label='CR_colorbar')
    cb = fig.colorbar(ScalarMappable(norm=cr_norm, cmap=cr_cmap), cax=cr_bar, orientation='vertical', ticks=[15, 30, 45, 60, 70], extend='max', alpha=0.82)
    cb.set_label('CR (dBZ)', fontsize=14 * font_scale, labelpad=10)
    cb.ax.tick_params(labelsize=12 * font_scale, length=3, pad=5)
    cb.outline.set_linewidth(0.6)
    if show_probability:
        probability_first_row = 2 if nrows >= 3 else 1
        probability_last_row = min(3, nrows - 1)
        probability_bar = fig.add_axes(colorbar_rect(probability_first_row, probability_last_row), label='Probability_colorbar')
        cb = fig.colorbar(ScalarMappable(norm=prob_norm, cmap=prob_cmap), cax=probability_bar, orientation='vertical', ticks=[0, 0.25, 0.5, 0.75, 1], alpha=0.84)
        cb.set_label('Surface hailfall probability', fontsize=14 * font_scale, labelpad=10)
        cb.ax.tick_params(labelsize=12 * font_scale, length=3, pad=5)
        cb.outline.set_linewidth(0.6)
    if save:
        if isinstance(save, (str, Path)):
            destination = Path(save).expanduser()
        else:
            destination = Path(save_dir or '.') / f'{case_id}_comparison_{display_interval:g}min.png'
        destination.parent.mkdir(parents=True, exist_ok=True)
        fig.savefig(destination, dpi=dpi, bbox_inches='tight', facecolor='white')
        print(f'Figure saved: {destination.resolve()}')
    if show:
        plt.show()
    return (fig, model_stats)


In [ ]:
case_id = CASE_ID
index = CASE_INDEX
title_name = CASE_TITLE
model_dic = {'HailRU': hailru, 'DIFF-T': difft, 'U-Net': tdunet}
output_dic = {}
input_data = np.load(train_dic_matrix[case_id][index]['input'])['arr_0']
mask_data = np.load(train_dic_matrix[case_id][index]['mask'])['arr_0']
output_data = np.load(train_dic_matrix[case_id][index]['output'])['arr_0']
input_tensor = torch.tensor(input_data, dtype=torch.float32, device=hailru.device).unsqueeze(0)
mask_tensor = torch.tensor(mask_data, dtype=torch.float32, device=hailru.device).unsqueeze(0).unsqueeze(0)
output_dic['Label'] = output_data
with torch.no_grad():
    for name in model_dic:
        prediction = model_dic[name](input_tensor) if name == 'DIFF-T' else model_dic[name](input_tensor, mask_tensor)
        output_dic[name] = prediction.cpu()[0].numpy()


In [ ]:
fig, model_stats = plot_model_comparison(output_dic, mask_dic, case_id=case_id, title=title_name, thre=0.5, data_dic=data_dic, sample_path=train_dic_matrix[case_id][index]['output'], input_len=10, sta_dic=sta_dic, city_boundary_path=CITY_BOUNDARY_PATH, station_obs=None, time_label_mode='actual', display_interval=12, cols_per_block=5, stats_scope='all', include_last=False, save=OUTPUT_ROOT / f'{case_id}_comparison.png')
